# Phase 1 - Data Understanding
**Project:** Credit Risk Modelling - Borrower Default Prediction & Credit Scoring
**Data:** LendingClub personal loans, issued 2007-06 to 2011-12 (`data/raw/loan.csv`)

This notebook is the readable walkthrough. All logic lives in `src/` so that the same functions are reused by scripts, later phases and the Streamlit app. Nothing is cleaned or dropped here; Phase 1 only measures.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

from src.data.load import load_raw
from src.data.dictionary import build_data_dictionary
from src.data.target import define_target, target_summary
from src.data import quality as q

df = load_raw()
q.dataset_overview(df)

## 1. Structure and data dictionary
`timing` is the leakage lens: only *Application* fields (and, by decision, *Origination* pricing fields) may enter a model that scores a borrower at origination.

In [ ]:
dictionary = build_data_dictionary(df)
dictionary[dictionary.missing_pct < 100][["column", "group", "timing", "missing_pct", "n_unique", "recommendation"]]

In [ ]:
dictionary.groupby(["group", "timing"]).size().rename("columns").reset_index()

## 2. Missing values

In [ ]:
missing = q.missing_value_report(df)
print(missing["band"].value_counts().sort_index())
missing[(missing.missing_pct > 0) & (missing.missing_pct < 100)]

## 3. Duplicates and constant columns

In [ ]:
print(q.duplicate_report(df))
q.constant_columns(df)

## 4. Formats, dates and value ranges

In [ ]:
q.format_issues(df)

In [ ]:
q.date_checks(df)

In [ ]:
q.numeric_profile(df, ["loan_amnt", "annual_inc", "dti", "int_rate", "installment", "revol_bal",
                       "revol_util", "open_acc", "total_acc", "delinq_2yrs", "inq_last_6mths", "pub_rec"])

In [ ]:
q.category_profile(df, ["term", "grade", "emp_length", "home_ownership", "verification_status", "purpose"])

## 5. Target variable
`Charged Off = 1`, `Fully Paid = 0`, `Current` excluded (outcome unknown).

In [ ]:
display(target_summary(df))
resolved = define_target(df)
print(f"Resolved loans: {len(resolved):,} | Charged-off rate: {resolved['default_spec'].mean():.4f}")

## 6. Outcome maturity
Chronological splitting puts the newest loans in the test set. If those cohorts still contain unresolved loans, excluding them changes the observed default rate.

In [ ]:
q.maturity_report(df)

In [ ]:
display(q.current_loans_age(df))
q.default_timing(df)

## 7. Leakage evidence
Repayment fields separate the classes almost perfectly because they *record* the outcome.

In [ ]:
q.leakage_evidence(df)

## 8. Is missingness informative?

In [ ]:
q.missingness_vs_default(df, ["mths_since_last_delinq", "mths_since_last_record", "emp_length",
                               "emp_title", "pub_rec_bankruptcies", "revol_util"])

## Phase 1 conclusions
See `reports/phase1/data_quality_report.md` for the full report and `data_dictionary.csv` for column-level decisions. Decisions taken for Phase 2 onwards:
1. LendingClub's `grade` / `sub_grade` / `int_rate` / `installment` are **excluded** from the models and used only as a benchmark.
2. Outcome maturity: the modelling target is **default within 36 months** - a charged-off loan whose arrears began within its first 36 months on book (onset estimated from the amortisation schedule and the last payment date; see `src/preprocessing.py`). Every loan has 53+ months of history, so `Current` loans are kept as goods and no cohort is biased. The original status-based definition is kept for a sensitivity check in Phase 10.